# 03 — Structured Feature Engineering

**Phase 1 — Data Engineering | MVP ladder: pre-MVP**

**Goal:** Engineer the 15-dimensional structured feature vector that feeds Branch C of the multimodal architecture, then prune any feature that fails the correlation gate (|corr(feat, T1)| < 0.02).

### 15-feature spec (per scope MD §8.4)

| # | Feature | Type | Source |
|---|---------|------|--------|
| 1 | `text_len_chars`     | numeric | tweet_text length in chars |
| 2 | `n_hashtags`         | numeric | tweet_text regex `#\w+` |
| 3 | `n_mentions`         | numeric | tweet_text regex `@\w+` |
| 4 | `url_present`        | binary  | tweet_text contains `https?://...` |
| 5 | `n_emoji`            | numeric | tweet_text emoji-range chars |
| 6 | `allcaps_frac`       | numeric | upper letters / all letters |
| 7 | `n_exclamations`     | numeric | count of `!` |
| 8 | `n_questions`        | numeric | count of `?` |
| 9 | `repeated_chars`     | numeric | count of runs of 3+ same letters (e.g. "heyyyy") |
| 10 | `profanity_count`   | numeric | better-profanity lexicon match count |
| 11 | `hate_keyword_count` | numeric | matches against `hatespeech_keywords.txt` |
| 12 | `vader_neg`          | numeric | VADER negative sentiment score |
| 13 | `vader_neu`          | numeric | VADER neutral sentiment score |
| 14 | `ocr_len`            | numeric | length of OCR text from `img_txt/{id}.json` |
| 15 | `ocr_present`        | binary  | OCR text non-empty |

### Correlation gate
After engineering, compute `|corr(feature, T1)|` for every feature. Drop any with |corr| < 0.02 — they add noise to the structured branch. Survivors are saved to `data/processed/structured_features.csv` keyed on `tweet_id`.


In [1]:
import ast
import json
import re
import random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from better_profanity import profanity

random.seed(42)
np.random.seed(42)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR     = PROJECT_ROOT / "data" / "MMHS150K"
PROCESSED    = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR  = PROJECT_ROOT / "outputs"

OCR_DIR       = DATA_DIR / "img_txt"
KEYWORDS_FILE = DATA_DIR / "hatespeech_keywords.txt"

# Load processed labels (we only need tweet_id, tweet_text, T1 for engineering + audit)
df = pd.read_csv(
    PROCESSED / "labels_parsed.csv",
    converters={"labels": ast.literal_eval, "labels_str": ast.literal_eval},
    dtype={"tweet_id": str, "img_filename": str},
)
df["T1"] = df["T1"].astype("int8")

print("df.shape:", df.shape)
print("Nulls in tweet_text/T1:",
      "tweet_text=", int(df["tweet_text"].isna().sum()),
      "T1=", int(df["T1"].isna().sum()))

# --- Initialise NLP tools ---
analyzer = SentimentIntensityAnalyzer()
profanity.load_censor_words()

# Read the bundled wordlist file directly. better-profanity's CENSOR_WORDSET
# wraps each word in a VaryingString class (for leetspeak fuzzy matching) which
# breaks re.escape. The raw wordlist gives plain strings — exact-match is what
# we want for tweet-level counting anyway.
import importlib.resources as resources
PROFANITY_WORDS = []
try:
    with resources.files("better_profanity").joinpath("profanity_wordlist.txt").open("r", encoding="utf-8") as f:
        PROFANITY_WORDS = [w.strip() for w in f if w.strip()]
except Exception as e:
    print(f"WARNING: could not load better-profanity lexicon ({e}); profanity_count will be all zeros")
print(f"Profanity lexicon size: {len(PROFANITY_WORDS)} words")

# --- Load hate-speech keywords from MMHS150K dataset ---
HATE_KEYWORDS = []
with open(KEYWORDS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        kw = line.strip().lower()
        if kw:
            HATE_KEYWORDS.append(kw)
print(f"Hate keywords loaded: {len(HATE_KEYWORDS)} terms (from hatespeech_keywords.txt)")
print("First 5:", HATE_KEYWORDS[:5])


df.shape: (149819, 10)
Nulls in tweet_text/T1: tweet_text= 0 T1= 0
Profanity lexicon size: 916 words
Hate keywords loaded: 86 terms (from hatespeech_keywords.txt)
First 5: ['asian drive', 'feminazi', 'sjw', 'womenagainstfeminism', 'blameonenotall']


In [2]:
# Bulk-load OCR text for all 149,819 rows. Threaded I/O cuts runtime 5-10x
# vs sequential reads on Windows NTFS.

def read_ocr(tid):
    p = OCR_DIR / f"{tid}.json"
    if not p.exists():
        return ""
    try:
        with open(p, "r", encoding="utf-8") as f:
            return json.load(f).get("img_text", "") or ""
    except Exception:
        return ""

with ThreadPoolExecutor(max_workers=16) as ex:
    ocr_texts = list(tqdm(
        ex.map(read_ocr, df["tweet_id"]),
        total=len(df),
        desc="Reading OCR JSONs",
    ))

df["ocr_text"] = ocr_texts
print(f"\nOCR text loaded for {len(df):,} rows.")
print(f"OCR-empty count: {(df['ocr_text'].str.len() == 0).sum():,}")
print(f"OCR-present rate: {(df['ocr_text'].str.len() > 0).mean()*100:.1f}%")
print("\ndf.shape:", df.shape, "| ocr_text nulls:", int(df["ocr_text"].isna().sum()))


Reading OCR JSONs:   0%|          | 0/149819 [00:00<?, ?it/s]


OCR text loaded for 149,819 rows.
OCR-empty count: 90,570
OCR-present rate: 39.5%

df.shape: (149819, 11) | ocr_text nulls: 0


In [3]:
# Engineer 11 text features via vectorised pandas.str ops.

URL_RE        = re.compile(r"https?://\S+")
HASHTAG_RE    = re.compile(r"#\w+")
MENTION_RE    = re.compile(r"@\w+")
EMOJI_RE = re.compile(
    "["
    "\U0001F1E0-\U0001F1FF"   # regional indicators (flags)
    "\U0001F300-\U0001F5FF"   # symbols & pictographs
    "\U0001F600-\U0001F64F"   # emoticons
    "\U0001F680-\U0001F6FF"   # transport & map symbols
    "\U0001F700-\U0001F77F"   # alchemical symbols
    "\U0001F780-\U0001F7FF"   # geometric extended
    "\U0001F800-\U0001F8FF"   # supplemental arrows-C
    "\U0001F900-\U0001F9FF"   # supplemental symbols & pictographs
    "\U0001FA00-\U0001FA6F"   # chess symbols
    "\U0001FA70-\U0001FAFF"   # symbols & pictographs extended-A
    "\U00002600-\U000027BF"   # misc symbols and dingbats
    "]"
)
REPEATED_RE   = re.compile(r"([A-Za-z])\1{2,}")     # 3+ same letters in a row

# Hate keyword regex: word-boundary match against full lexicon
HATE_RE = re.compile(
    r"\b(?:" + "|".join(re.escape(k) for k in HATE_KEYWORDS) + r")\b",
    flags=re.IGNORECASE,
)
# Profanity regex (single big regex for speed)
if PROFANITY_WORDS:
    PROF_RE = re.compile(
        r"\b(?:" + "|".join(re.escape(w) for w in PROFANITY_WORDS) + r")\b",
        flags=re.IGNORECASE,
    )
else:
    PROF_RE = None

# 1. text_len_chars
df["text_len_chars"] = df["tweet_text"].str.len().astype("int32")

# 2. n_hashtags
df["n_hashtags"] = df["tweet_text"].str.count(HASHTAG_RE).astype("int16")

# 3. n_mentions
df["n_mentions"] = df["tweet_text"].str.count(MENTION_RE).astype("int16")

# 4. url_present (binary)
df["url_present"] = df["tweet_text"].str.contains(URL_RE).astype("int8")

# 5. n_emoji
df["n_emoji"] = df["tweet_text"].str.count(EMOJI_RE).astype("int16")

# 6. allcaps_frac (fraction of alphabetic chars that are uppercase)
upper_count = df["tweet_text"].str.count(r"[A-Z]")
alpha_count = df["tweet_text"].str.count(r"[A-Za-z]")
df["allcaps_frac"] = (upper_count / alpha_count.replace(0, np.nan)).fillna(0.0).astype("float32")

# 7. n_exclamations
df["n_exclamations"] = df["tweet_text"].str.count(r"!").astype("int16")

# 8. n_questions
df["n_questions"] = df["tweet_text"].str.count(r"\?").astype("int16")

# 9. repeated_chars (count of runs of 3+ same letter)
df["repeated_chars"] = df["tweet_text"].str.count(REPEATED_RE).astype("int16")

# 10. profanity_count
if PROF_RE is not None:
    df["profanity_count"] = df["tweet_text"].str.count(PROF_RE).astype("int16")
else:
    df["profanity_count"] = pd.Series(0, index=df.index, dtype="int16")

# 11. hate_keyword_count
df["hate_keyword_count"] = df["tweet_text"].str.count(HATE_RE).astype("int16")

cols_added = ["text_len_chars", "n_hashtags", "n_mentions", "url_present",
              "n_emoji", "allcaps_frac", "n_exclamations", "n_questions",
              "repeated_chars", "profanity_count", "hate_keyword_count"]
print(f"Added {len(cols_added)} regex-based features.")
print("\ndf.shape:", df.shape)
print("Nulls in new cols:", int(df[cols_added].isna().sum().sum()))
print("\nSample (first 3 rows):")
print(df[["tweet_id"] + cols_added].head(3).to_string())


Added 11 regex-based features.

df.shape: (149819, 22)
Nulls in new cols: 0

Sample (first 3 rows):
              tweet_id  text_len_chars  n_hashtags  n_mentions  url_present  n_emoji  allcaps_frac  n_exclamations  n_questions  repeated_chars  profanity_count  hate_keyword_count
0  1114679353714016256              44           0           1            1        0      0.222222               0            0               0                1                   1
1  1063020048816660480              46           0           0            1        0      0.142857               0            0               0                1                   1
2  1108927368075374593              80           0           0            1        0      0.761905               0            0               0                3                   2


In [4]:
# VADER neg + neu sentiment scores.
# Threaded with ThreadPoolExecutor(16) for consistency with the OCR loader pattern.
# Note: vaderSentiment is pure Python so the GIL limits how much threading actually
# helps; real-world speedup is usually modest (some C-level ops in `re` release the GIL).
# Wall-clock time for 150K rows is reported by tqdm at the end of the run.

from concurrent.futures import ThreadPoolExecutor

def vader_score(txt):
    s = analyzer.polarity_scores(txt)
    return s["neg"], s["neu"]

with ThreadPoolExecutor(max_workers=16) as ex:
    vader_results = list(tqdm(
        ex.map(vader_score, df["tweet_text"]),
        total=len(df),
        desc="VADER scoring (threaded x16)",
    ))

vader_df = pd.DataFrame(vader_results, columns=["vader_neg", "vader_neu"], index=df.index)
df["vader_neg"] = vader_df["vader_neg"].astype("float32")
df["vader_neu"] = vader_df["vader_neu"].astype("float32")

print(f"\nAdded vader_neg, vader_neu.")
print("\ndf.shape:", df.shape)
print("Nulls:", df[["vader_neg", "vader_neu"]].isna().sum().to_dict())
print("\nVADER summary stats:")
print(df[["vader_neg", "vader_neu"]].describe().round(4))


VADER scoring (threaded x16):   0%|          | 0/149819 [00:00<?, ?it/s]


Added vader_neg, vader_neu.

df.shape: (149819, 24)
Nulls: {'vader_neg': 0, 'vader_neu': 0}

VADER summary stats:
         vader_neg    vader_neu
count  149819.0000  149819.0000
mean        0.1748       0.7279
std         0.1876       0.2071
min         0.0000       0.0400
25%         0.0000       0.5750
50%         0.1470       0.7190
75%         0.2990       1.0000
max         0.9600       1.0000


In [5]:
# OCR-derived features (already loaded df["ocr_text"] in Cell 3).

df["ocr_len"]     = df["ocr_text"].str.len().astype("int32")
df["ocr_present"] = (df["ocr_len"] > 0).astype("int8")

print("Added ocr_len, ocr_present.")
print("\ndf.shape:", df.shape)
print("Nulls:", df[["ocr_len", "ocr_present"]].isna().sum().to_dict())
print(f"\nOCR present rate: {df['ocr_present'].mean()*100:.1f}%")
print("OCR length percentiles:")
print(df["ocr_len"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(1))


Added ocr_len, ocr_present.

df.shape: (149819, 26)
Nulls: {'ocr_len': 0, 'ocr_present': 0}

OCR present rate: 39.5%
OCR length percentiles:
count    149819.0
mean         31.4
std          84.3
min           0.0
50%           0.0
90%          95.0
95%         181.0
99%         391.0
max        3128.0
Name: ocr_len, dtype: float64


In [6]:
# Correlation audit -> drop |corr|<0.02 -> save survivors to CSV.

FEATURES = [
    "text_len_chars", "n_hashtags", "n_mentions", "url_present",
    "n_emoji", "allcaps_frac", "n_exclamations", "n_questions",
    "repeated_chars", "profanity_count", "hate_keyword_count",
    "vader_neg", "vader_neu", "ocr_len", "ocr_present",
]
assert len(FEATURES) == 15, f"Expected 15 features, got {len(FEATURES)}"

# Pearson correlation with T1 (binary 0/1).
# pandas auto-casts ints to float for corr.
corrs = pd.Series({f: df[f].corr(df["T1"]) for f in FEATURES}, dtype="float32")

audit = pd.DataFrame({
    "feature": corrs.index,
    "corr_T1": corrs.values,
    "abs_corr": corrs.abs().values,
})
audit = audit.sort_values("abs_corr", ascending=False).reset_index(drop=True)
audit["keep"] = audit["abs_corr"] >= 0.02

print("=" * 70)
print("CORRELATION AUDIT — sorted by |corr(feature, T1)| desc")
print("=" * 70)
print(audit.round(4).to_string(index=False))

kept    = audit.loc[audit["keep"], "feature"].tolist()
dropped = audit.loc[~audit["keep"], "feature"].tolist()

print("\n" + "=" * 70)
print(f"KEPT    ({len(kept):>2}): {kept}")
print(f"DROPPED ({len(dropped):>2}): {dropped}")
print("=" * 70)

# --- Save survivors keyed on tweet_id ---
SAVE_COLS = ["tweet_id"] + kept
out_df = df[SAVE_COLS].copy()
out_path = PROCESSED / "structured_features.csv"
out_df.to_csv(out_path, index=False)
size_mb = out_path.stat().st_size / 1024 / 1024

print(f"\nSaved -> {out_path}  ({size_mb:.1f} MB)")
print(f"  shape: {out_df.shape}  (rows × cols)")
print(f"  nulls per column:")
print(out_df.isna().sum().to_string())

print("\n--- Final dtypes ---")
print(out_df.dtypes.to_string())

print("\n--- Per-feature summary statistics ---")
print(out_df.drop(columns=["tweet_id"]).describe().round(3).T.to_string())


CORRELATION AUDIT — sorted by |corr(feature, T1)| desc
           feature  corr_T1  abs_corr  keep
         vader_neg   0.1714    0.1714  True
         vader_neu  -0.1326    0.1326  True
           n_emoji  -0.0478    0.0478  True
hate_keyword_count   0.0328    0.0328  True
       ocr_present   0.0277    0.0277  True
   profanity_count   0.0275    0.0275  True
        n_hashtags   0.0272    0.0272  True
           ocr_len   0.0271    0.0271  True
        n_mentions   0.0234    0.0234  True
    repeated_chars  -0.0178    0.0178 False
      allcaps_frac   0.0131    0.0131 False
    text_len_chars   0.0095    0.0095 False
    n_exclamations   0.0024    0.0024 False
       n_questions   0.0014    0.0014 False
       url_present      NaN       NaN False

KEPT    ( 9): ['vader_neg', 'vader_neu', 'n_emoji', 'hate_keyword_count', 'ocr_present', 'profanity_count', 'n_hashtags', 'ocr_len', 'n_mentions']
DROPPED ( 6): ['repeated_chars', 'allcaps_frac', 'text_len_chars', 'n_exclamations', 'n_quest

D:\Anaconda\envs\cyberbully_project\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
D:\Anaconda\envs\cyberbully_project\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]



Saved -> D:\Cyberbullying Detection\data\processed\structured_features.csv  (6.6 MB)
  shape: (149819, 10)  (rows × cols)
  nulls per column:
tweet_id              0
vader_neg             0
vader_neu             0
n_emoji               0
hate_keyword_count    0
ocr_present           0
profanity_count       0
n_hashtags            0
ocr_len               0
n_mentions            0

--- Final dtypes ---
tweet_id                  str
vader_neg             float32
vader_neu             float32
n_emoji                 int16
hate_keyword_count      int16
ocr_present              int8
profanity_count         int16
n_hashtags              int16
ocr_len                 int32
n_mentions              int16

--- Per-feature summary statistics ---
                       count    mean     std   min    25%    50%     75%      max
vader_neg           149819.0   0.175   0.188  0.00  0.000  0.147   0.299     0.96
vader_neu           149819.0   0.728   0.207  0.04  0.575  0.719   1.000     1.00
n_emoji  

## Note — `hate_keyword_count` correlation is confounded by dataset construction

The 86 hate keywords used in this feature come directly from MMHS150K's `hatespeech_keywords.txt`, which is the same Hatebase term list Gomez et al. (2019) used to **seed the dataset during collection** — every tweet in MMHS150K was selected because it matched at least one of these terms.

**Consequence:**
- Mean `hate_keyword_count` ≈ 0.98, std ≈ 0.27 — nearly every tweet has exactly one match.
- The feature has near-zero variance, so its raw correlation with T1 (+0.033) is **structurally inflated by the sampling pipeline**, not by an independent semantic signal.
- A model that learns from this feature would essentially be learning a constant.

**Decision:** kept for completeness rather than dropped — pruning a dataset-level artefact in addition to noise-level features risks over-aggressive feature reduction. Its correlation should be interpreted with this confound in mind. **This will be discussed in the report's Limitations section.**

---

## Note — `url_present` dropped due to zero variance

`url_present` was dropped during the correlation audit because every MMHS150K tweet contains a Twitter `t.co` URL wrapper, making the feature effectively constant (correlation with T1 = NaN due to zero variance in the denominator). This is a property of Twitter's link handling, not a property of hate speech. The feature is preserved in the engineering code for future use on non-Twitter datasets where it may carry signal.
